# Building Iterative Multi-Agent Systems with Hugging Face Inference + LangGraph
## Task → Evaluation → Continuous Improvement Loop → Multi-Agent Orchestration

**Hands-on course notebook** (November 2026)

This version uses:
- **Hugging Face Inference API** (`huggingface_hub.InferenceClient`) for real LLM calls
- **LangGraph** for the stateful cyclic Generator-Critic loop

### Setup
1. Get a Hugging Face token: https://huggingface.co/settings/tokens
2. Set it below (or as an environment variable `HF_TOKEN`)
3. Run the cells in order

In [ ]:
# Install if needed (uncomment on first run)
# !pip install langgraph huggingface_hub -q

import os
from typing import TypedDict, Annotated
from huggingface_hub import InferenceClient
from langgraph.graph import StateGraph, END

# --- Configuration ---
HF_TOKEN = os.getenv("HF_TOKEN", "hf_your_token_here")  # <-- paste your token or set env var
MODEL = "Qwen/Qwen2.5-7B-Instruct"  # good free-tier option; change as needed

client = InferenceClient(token=HF_TOKEN)
print(f"Using model: {MODEL}")

## Module 1: Define the Task

In [ ]:
TASK = """
Write a concise technical summary of this concept:

Concept: Using cyclic state graphs (LangGraph-style) to implement a Generator-Critic loop for iterative improvement of LLM outputs until a quality threshold is met.

Requirements:
- 250-400 words
- Clear sections: Problem, Approach, Benefits, Limitations, Implications
- Accurate and practical
- End with 2 concrete next steps a practitioner should take
"""

print("Task ready.")

## Module 2: Generator and Evaluator (Hugging Face Inference)

In [ ]:
def generate(task: str, feedback: str = "") -> str:
    """Generator agent using HF Inference."""
    messages = [
        {"role": "system", "content": "You are a precise technical writer. Follow the user's requirements exactly."},
        {"role": "user", "content": task + (f"\n\nPrevious feedback to address:\n{feedback}" if feedback else "")}
    ]
    response = client.chat_completion(
        messages=messages,
        model=MODEL,
        max_tokens=600,
        temperature=0.7,
    )
    return response.choices[0].message.content.strip()


def evaluate(task: str, output: str) -> dict:
    """Evaluator agent (LLM-as-judge) using HF Inference."""
    prompt = f"""You are a strict technical reviewer.

Task:
{task}

Output to evaluate:
{output}

Score the output from 1 to 10 on overall quality against the requirements.
List specific issues if any.
Respond ONLY with valid JSON in this exact format:
{{
  "score": 7.5,
  "passed": false,
  "issues": ["issue 1", "issue 2"],
  "feedback": "Clear, actionable feedback for the next revision."
}}
"""
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL,
        max_tokens=400,
        temperature=0.2,
    )
    text = response.choices[0].message.content.strip()
    # Basic JSON extraction
    import json, re
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
    # Fallback
    return {"score": 5.0, "passed": False, "issues": ["Parse error"], "feedback": text[:300]}

## Module 3: Iterative Loop with LangGraph

In [ ]:
class State(TypedDict):
    task: str
    output: str
    feedback: str
    score: float
    attempts: int
    max_attempts: int
    history: list


def generator_node(state: State) -> dict:
    print(f"\n--- Generator (attempt {state['attempts'] + 1}) ---")
    output = generate(state["task"], state.get("feedback", ""))
    print(output[:300] + "..." if len(output) > 300 else output)
    return {
        "output": output,
        "attempts": state["attempts"] + 1,
    }


def evaluator_node(state: State) -> dict:
    print("\n--- Evaluator ---")
    result = evaluate(state["task"], state["output"])
    score = float(result.get("score", 5.0))
    feedback = result.get("feedback", "")
    passed = result.get("passed", score >= 8.0)
    print(f"Score: {score} | Passed: {passed}")
    print(f"Feedback: {feedback}")

    history = state.get("history", []) + [{
        "attempt": state["attempts"],
        "score": score,
        "feedback": feedback,
    }]

    return {
        "score": score,
        "feedback": feedback,
        "history": history,
    }


def router(state: State) -> str:
    if state["score"] >= 8.0:
        return "accept"
    if state["attempts"] >= state["max_attempts"]:
        return "max_reached"
    return "revise"


# Build the graph
workflow = StateGraph(State)
workflow.add_node("generator", generator_node)
workflow.add_node("evaluator", evaluator_node)

workflow.set_entry_point("generator")
workflow.add_edge("generator", "evaluator")
workflow.add_conditional_edges(
    "evaluator",
    router,
    {
        "accept": END,
        "max_reached": END,
        "revise": "generator",
    },
)

app = workflow.compile()
print("LangGraph app compiled.")

In [ ]:
# Run the iterative loop
initial_state = {
    "task": TASK,
    "output": "",
    "feedback": "",
    "score": 0.0,
    "attempts": 0,
    "max_attempts": 3,
    "history": [],
}

result = app.invoke(initial_state)

print("\n" + "="*60)
print("FINAL OUTPUT")
print("="*60)
print(result["output"])
print("\nScore history:", [h["score"] for h in result["history"]])

## Module 4: Multi-Agent Notes

The same pattern extends to multiple specialists:
- Add a `research` node before `generator`
- Router can send back to research *or* generator depending on the critic's feedback
- Shared `State` carries notes, draft, score, and attempt count

LangGraph makes the cycle and the routing decision explicit and safe (hard max attempts, checkpoints available).

## Production Tips
- Always keep a hard `max_attempts`
- Log full history for debugging
- Consider a stronger model for the evaluator than the generator
- Add stagnation detection (same score twice → stop)
- Use structured outputs / JSON mode when the model supports it
- For free-tier rate limits, add small delays or use a paid Inference Endpoint